
# Notebook de criptografía  `cryptography`


In [1]:

# Lib Crypto Hazmat (hazardous materials / materiales peligrosos :D ) 
!pip install cryptography

import os
import base64

from cryptography.hazmat.primitives import hashes, padding as sym_padding
from cryptography.hazmat.primitives.asymmetric import rsa, padding, ec
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

# En versiones recientes, 3DES está en el módulo "decrepit"
from cryptography.hazmat.decrepit.ciphers import algorithms as decrepit_algorithms

from cryptography.exceptions import InvalidSignature

def b64(data: bytes) -> str:
    return base64.b64encode(data).decode("utf-8")

def linea(titulo: str):
    print("\n" + "=" * 80)
    print(titulo)
    print("=" * 80)



## A. Generar un Resumen Digital (Message Digest / Hash)

En este ejemplo se calcula un hash **SHA-256** de un mensaje.


In [2]:

linea("A. RESUMEN DIGITAL / HASH")

mensaje = b"Este es un mensaje de prueba para generar un resumen digital."

digestor = hashes.Hash(hashes.SHA256())
digestor.update(mensaje)
resumen = digestor.finalize()

print("Mensaje original:", mensaje.decode("utf-8"))
print("SHA-256 (hex):", resumen.hex())
print("Longitud del hash:", len(resumen), "bytes")



A. RESUMEN DIGITAL / HASH
Mensaje original: Este es un mensaje de prueba para generar un resumen digital.
SHA-256 (hex): e6e24ebccf1f1389c660d014777cf794add39df0c3f8787b7d4f36bfad989910
Longitud del hash: 32 bytes



## B. Generar una Firma Digital con RSA (Firmar y verificar)

La documentación oficial recomienda usar un padding apropiado para firmas RSA. En este notebook usamos **RSA + PSS + SHA-256** para firmar y verificar. 


In [3]:

linea("B. FIRMA DIGITAL RSA")

# 1. Generar par de llaves RSA
clave_privada_rsa = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)
clave_publica_rsa = clave_privada_rsa.public_key()

# 2. Mensaje a firmar
mensaje_firma = b"Mensaje firmado digitalmente con RSA."

# 3. Firmar
firma_rsa = clave_privada_rsa.sign(
    mensaje_firma,
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)

print("Mensaje:", mensaje_firma.decode())
print("Firma RSA (Base64):", b64(firma_rsa))

# 4. Verificar
try:
    clave_publica_rsa.verify(
        firma_rsa,
        mensaje_firma,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("Verificación correcta: la firma es válida.")
except InvalidSignature:
    print("La firma NO es válida.")



B. FIRMA DIGITAL RSA
Mensaje: Mensaje firmado digitalmente con RSA.
Firma RSA (Base64): C3B5L2wzvzGuWXk6BySLK/Sf/Fudi2gX4dwb9OrHcBsLCLUDOqSruiTEf0vn1/cgGAreicXn0jgHYnev+L9jOxX7hx/neFRf2gtwbPuERuo4kmWEzF03onrVDUQxXc+g2fdoa26Ti7LY+uHrTw2MQFTBY1YmK60FnzOo/hVZyTsodPKAPMvqwJQKBhx5W8DlzUFLXyEKkcL/Jrdk/Jf8ycwIfVMn4hjbiZh7XRSfXu2WQGbqjdMcDH4rT5F2Uwjn9uUzsorusBM84CPqheDPpKYfnl0N88ILuNQp4WgliqzXMlCXxMv7xMZHJosXq4394EwEjZbI+Bef7KnJiZnEvw==
Verificación correcta: la firma es válida.



## C. Cifrar y descifrar un mensaje con criptografía simétrica

Aquí se muestran dos ejemplos:

- **3DES en modo CBC** con padding PKCS7.
- **AES en modo CBC** con padding PKCS7.

> Para cifrado moderno, suele preferirse un modo autenticado como **AES-GCM**. Aquí se usa CBC porque es didáctico para mostrar el flujo clásico de cifrar y descifrar con bloques.


In [4]:

def cifrar_simetrico_cbc(algoritmo, clave: bytes, mensaje: bytes):
    iv = os.urandom(algoritmo.block_size // 8)
    cipher = Cipher(algoritmo(clave), modes.CBC(iv))
    encryptor = cipher.encryptor()

    padder = sym_padding.PKCS7(algoritmo.block_size).padder()
    mensaje_padded = padder.update(mensaje) + padder.finalize()

    ciphertext = encryptor.update(mensaje_padded) + encryptor.finalize()
    return iv, ciphertext

def descifrar_simetrico_cbc(algoritmo, clave: bytes, iv: bytes, ciphertext: bytes):
    cipher = Cipher(algoritmo(clave), modes.CBC(iv))
    decryptor = cipher.decryptor()

    mensaje_padded = decryptor.update(ciphertext) + decryptor.finalize()

    unpadder = sym_padding.PKCS7(algoritmo.block_size).unpadder()
    mensaje = unpadder.update(mensaje_padded) + unpadder.finalize()
    return mensaje


In [5]:

linea("C1. CIFRADO SIMÉTRICO CON 3DES")

mensaje_3des = b"Mensaje secreto con 3DES para fines academicos."
# 3DES acepta 8, 16 o 24 bytes; 24 bytes = Triple DES completo
clave_3des = os.urandom(24)

iv_3des, cifrado_3des = cifrar_simetrico_cbc(
    decrepit_algorithms.TripleDES,
    clave_3des,
    mensaje_3des
)

descifrado_3des = descifrar_simetrico_cbc(
    decrepit_algorithms.TripleDES,
    clave_3des,
    iv_3des,
    cifrado_3des
)

print("Mensaje original:", mensaje_3des.decode())
print("Clave 3DES (Base64):", b64(clave_3des))
print("IV 3DES (Base64):", b64(iv_3des))
print("Texto cifrado 3DES (Base64):", b64(cifrado_3des))
print("Texto descifrado:", descifrado_3des.decode())
print("¿Coincide?:", mensaje_3des == descifrado_3des)



C1. CIFRADO SIMÉTRICO CON 3DES
Mensaje original: Mensaje secreto con 3DES para fines academicos.
Clave 3DES (Base64): ITt2yB1m3uFwKjNLqxRaBksGccDm+g+w
IV 3DES (Base64): AsgpuBt3ZSc=
Texto cifrado 3DES (Base64): qgd2oRnUKP17/6+0IF8VN2EznMqnOg4QogDrcY7Ha3g0zAlfnos9Vgw3epOLe6EX
Texto descifrado: Mensaje secreto con 3DES para fines academicos.
¿Coincide?: True


In [6]:

linea("C2. CIFRADO SIMÉTRICO CON AES")

mensaje_aes = b"Mensaje secreto con AES."
clave_aes = os.urandom(32)  # AES-256

iv_aes, cifrado_aes = cifrar_simetrico_cbc(
    algorithms.AES,
    clave_aes,
    mensaje_aes
)

descifrado_aes = descifrar_simetrico_cbc(
    algorithms.AES,
    clave_aes,
    iv_aes,
    cifrado_aes
)

print("Mensaje original:", mensaje_aes.decode())
print("Clave AES-256 (Base64):", b64(clave_aes))
print("IV AES (Base64):", b64(iv_aes))
print("Texto cifrado AES (Base64):", b64(cifrado_aes))
print("Texto descifrado:", descifrado_aes.decode())
print("¿Coincide?:", mensaje_aes == descifrado_aes)



C2. CIFRADO SIMÉTRICO CON AES
Mensaje original: Mensaje secreto con AES.
Clave AES-256 (Base64): FfU/V0+SwAnj91ZSBh4CN1TuKKgM4xu0TRrqHZpBbjE=
IV AES (Base64): +zE2mU+tgKQIxTSlGV63HQ==
Texto cifrado AES (Base64): oto7ffH12djfHUPUpiYbHlb3C7eCIrFyrhLJvOZEd0k=
Texto descifrado: Mensaje secreto con AES.
¿Coincide?: True



## D. Cifrar y descifrar con criptografía asimétrica de llave pública

Para cifrado RSA, la documentación oficial recomienda **OAEP** como padding de cifrado. Aquí se usa **RSA + OAEP + SHA-256**. 


In [7]:

linea("D. CIFRADO / DESCIFRADO ASIMÉTRICO RSA")

mensaje_rsa = b"Mensaje confidencial cifrado con llave publica RSA."

# Reutilizamos el par de llaves RSA generado antes
ciphertext_rsa = clave_publica_rsa.encrypt(
    mensaje_rsa,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

mensaje_descifrado_rsa = clave_privada_rsa.decrypt(
    ciphertext_rsa,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

print("Mensaje original:", mensaje_rsa.decode())
print("Texto cifrado RSA (Base64):", b64(ciphertext_rsa))
print("Texto descifrado RSA:", mensaje_descifrado_rsa.decode())
print("¿Coincide?:", mensaje_rsa == mensaje_descifrado_rsa)



D. CIFRADO / DESCIFRADO ASIMÉTRICO RSA
Mensaje original: Mensaje confidencial cifrado con llave publica RSA.
Texto cifrado RSA (Base64): I9oBFBtaydexQvFUgnqvIK6zUWbXKVB+iwje4xnTJ4QIlrHsd3T5uQTY3gM/LLMaOVsDNK+YiS/RL8fuld6nupLye+BIbBL8gnjHhR56QQ3By8SWryhLDXoPZe+zsymuhUWIx9dtu5nYrYXCaSbX0SC2/4mQ9ze2nDjfKbgDiEU4ZQsqGxg+y1WFqxhXt0ZabM6SgUad6zJ+J3crBbARLemyYV6oAlVWOwm9fbZkrjYAYsEWKqvVZED3FufqtOhLbAs18uZ0oJFTxc9agXP82oNshYZaMOh+LrcL0ULD+My93jan7u2ErnbxqKycWuid27MiGvzwOAEA1V13K72Nuw==
Texto descifrado RSA: Mensaje confidencial cifrado con llave publica RSA.
¿Coincide?: True



## E. Firmar y verificar un mensaje usando Criptografía de Curvas Elípticas

En este ejemplo se usa **ECDSA** con la curva **SECP256R1** y hash **SHA-256**. La documentación oficial presenta ECDSA como el esquema de firma para llaves EC. 


In [8]:

linea("E. FIRMA Y VERIFICACIÓN CON CURVAS ELÍPTICAS (ECDSA)")

# 1. Generar llaves EC
clave_privada_ec = ec.generate_private_key(ec.SECP256R1())
clave_publica_ec = clave_privada_ec.public_key()

mensaje_ec = b"Mensaje firmado con criptografia de curvas elipticas."

# 2. Firmar
firma_ec = clave_privada_ec.sign(
    mensaje_ec,
    ec.ECDSA(hashes.SHA256())
)

print("Mensaje:", mensaje_ec.decode())
print("Firma ECDSA (Base64):", b64(firma_ec))

# 3. Verificar
try:
    clave_publica_ec.verify(
        firma_ec,
        mensaje_ec,
        ec.ECDSA(hashes.SHA256())
    )
    print("Verificación correcta: la firma ECDSA es válida.")
except InvalidSignature:
    print("La firma ECDSA NO es válida.")



E. FIRMA Y VERIFICACIÓN CON CURVAS ELÍPTICAS (ECDSA)
Mensaje: Mensaje firmado con criptografia de curvas elipticas.
Firma ECDSA (Base64): MEUCIEForYRdA9BlpOVMM99XUfi0XIZ/DeZ7whztTD8ljV//AiEAzFJae1IyGuqJFQ7BrbIKepuc3EisdXks2MgEHuVwUes=
Verificación correcta: la firma ECDSA es válida.
